# Park et al. (2020): Illinois student learning loss inputs
School-year mean daily maximum temperature for 102 Illinois counties, school years ending 1985–2094. Absolute °C; SSP2-4.5 and SSP3-7.0; one CSV per model/member/scenario.

Run this notebook on Keeling from the Notebooks folder alongside your existing `loca2_common.py`. It imports the shared county aggregator without modifying it. Requires numpy, pandas, xarray, dask, netCDF4, cftime, geopandas and tqdm. Existing outputs are skipped.

Source specification: `Weather_Data_Needs_Student_Learning_Loss.xlsx`, sheet `Park et al. (2020)`, rows 4–13. School year 1985 requires data from August 20, 1984. School year 2015 joins historical 2014 to scenario 2015. No humidity inputs or period averaging are used.

In [ ]:
from pathlib import Path
from contextlib import ExitStack
from datetime import timedelta
import os
import tempfile
import json
import numpy as np
import pandas as pd
import xarray as xr
import cftime
import geopandas as gpd
from tqdm.auto import tqdm
import loca2_common as common

INPUT_ROOT = Path('/data/keeling/a/tahsina2/b/Climate_Impact/Data/LOCA2_Illinois')
OUTPUT_ROOT = INPUT_ROOT / 'Park et al. (2020)'
COUNTY_SHAPEFILE = INPUT_ROOT.parent / 'Illinois_County_Boundaries' / 'IL_BNDY_County_Py.shp'
SCENARIOS = ('ssp245', 'ssp370')
FIRST_YEAR, LAST_YEAR = 1985, 2094
OVERWRITE = False
RUN_LIMIT = None  # Set to 1 for the first discovered run.
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
counties = gpd.read_file(COUNTY_SHAPEFILE).to_crs('EPSG:4326').reset_index(drop=True)
fips = counties['CO_FIPS'].astype(int).astype(str).str.zfill(3)
counties['county_id'] = fips.where(fips.str.len() == 5, '17' + fips)
counties['county_name'] = counties['COUNTY_NAM'].astype(str).str.strip()
if len(counties) != 102 or not counties.county_id.is_unique:
    raise ValueError('Expected 102 unique Illinois counties')
if not counties.county_id.str.match(r'^17\d{3}$').all():
    raise ValueError('Invalid Illinois county FIPS codes')


## Discover and load daily maximum temperature
Only daily gridded `tasmax` inputs are accepted. Summary products are excluded. Historical data must match the same member. Duplicate calendar dates, incompatible grids and unknown units stop the affected run rather than being silently resolved.

In [ ]:
def daily_files(folder):
    excluded = ('monthly', 'yearly', 'annual', 'seasonal', 'county_avg')
    return [p for p in sorted(folder.glob('*.nc'))
            if not any(tag in p.name.lower() for tag in excluded)]

def discover_runs():
    return sorted({(p.parent.name, p.name, scenario)
                   for scenario in SCENARIOS
                   for p in (INPUT_ROOT / 'tasmax' / scenario).glob('*/*')
                   if p.is_dir() and daily_files(p)})

def to_celsius(data):
    unit = str(data.attrs.get('units', '')).lower().strip().replace(' ', '').replace('_', '')
    if unit in {'k', 'kelvin', 'degreek', 'degreesk'}:
        result = data.astype('float64') - 273.15
    elif unit in {'c', 'degc', 'celsius', 'degreecelsius', 'degreescelsius', '°c'}:
        result = data.astype('float64')
    else:
        raise ValueError(f'Unsupported temperature units: {data.attrs.get("units")!r}')
    result.attrs = dict(data.attrs, units='degree_Celsius')
    return result

def load_run(stack, model, member, scenario):
    pieces, sources = [], []
    for experiment, start, end in (
        ('historical', '1984-08-20', '2014-12-31'),
        (scenario, '2015-01-01', '2094-06-15'),
    ):
        folder = INPUT_ROOT / 'tasmax' / experiment / model / member
        paths = daily_files(folder)
        if not paths:
            raise FileNotFoundError(f'No daily tasmax files: {folder}')
        for path in paths:
            ds = stack.enter_context(xr.open_dataset(path, chunks={'time': 366}, decode_timedelta=False))
            if 'tasmax' not in ds:
                raise KeyError(f'{path.name}: expected daily variable tasmax; found {list(ds.data_vars)}')
            data = ds['tasmax']
            data = data.rename({a:b for a,b in [('latitude','lat'),('longitude','lon')] if a in data.dims})
            if set(data.dims) != {'time', 'lat', 'lon'}:
                raise ValueError(f'Unexpected dimensions: {data.dims}')
            # Compare calendar-date strings so 360-day calendars do not need December 31.
            dates = data.time.dt.strftime('%Y-%m-%d')
            data = data.sel(time=(dates >= start) & (dates <= end))
            if not data.sizes['time']:
                continue
            if np.issubdtype(data.time.dtype, np.datetime64):
                daily_time = data.time.values.astype('datetime64[D]')
            else:
                daily_time = [t.replace(hour=0, minute=0, second=0, microsecond=0) for t in data.time.values]
            data = to_celsius(data).assign_coords(time=daily_time).sortby('lat').sortby('lon')
            pieces.append(data)
            sources.append(str(path))
    if not pieces:
        raise ValueError('No temperature records in requested range')
    calendars = {p.time.dt.calendar for p in pieces}
    if len(calendars) != 1:
        raise ValueError(f'Inconsistent calendars: {calendars}')
    data = xr.concat(pieces, dim='time', join='exact', coords='minimal', compat='equals').sortby('time')
    if not data.indexes['time'].is_unique:
        raise ValueError('Duplicate daily dates; inspect overlapping source files')
    return data, sources

def expected_dates(year, calendar_name):
    start = cftime.datetime(year - 1, 8, 20, calendar=calendar_name)
    end = cftime.datetime(year, 6, 15, calendar=calendar_name)
    return [(start + timedelta(days=i)).strftime('%Y-%m-%d')
            for i in range((end - start).days + 1)]


## Calculate complete school years
Daily county means use the shared cosine-latitude area weighting. Each included day receives equal temporal weight. Available grid cells contribute to each daily mean; valid-area fractions describe spatial coverage. A missing county-day makes that county's school-year mean NaN. Missing dates fail the run. No numeric coverage cutoff is imposed.

In [ ]:
def create_table(model, member, scenario):
    tables = []
    with ExitStack() as stack:
        tasmax, sources = load_run(stack, model, member, scenario)
        calendar_name = tasmax.time.dt.calendar
        for year in range(FIRST_YEAR, LAST_YEAR + 1):
            expected = expected_dates(year, calendar_name)
            dates = tasmax.time.dt.strftime('%Y-%m-%d')
            block = tasmax.sel(time=(dates >= expected[0]) & (dates <= expected[-1]))
            actual = block.time.dt.strftime('%Y-%m-%d').values.tolist()
            if actual != expected:
                missing = sorted(set(expected) - set(actual))
                raise ValueError(f'School year {year}: incomplete dates; missing {len(missing)}, examples {missing[:5]}')
            block = block.load()
            if bool(((block < -100) | (block > 70) | np.isinf(block)).any().item()):
                raise ValueError(f'School year {year}: invalid Celsius temperature range or infinity')
            grid = xr.Dataset({
                'tasmax_c': block,
                'valid_area_fraction': np.isfinite(block).astype('float64'),
            })
            daily = common.aggregate_to_illinois_counties(dataset=grid, counties=counties).load()
            if daily.sizes.get('county') != 102:
                raise ValueError('County aggregation did not return 102 counties')
            annual = xr.Dataset({
                'school_year_mean_tasmax_c': daily.tasmax_c.mean('time', skipna=False),
                'valid_days': np.isfinite(daily.tasmax_c).sum('time'),
                'mean_valid_area_fraction': daily.valid_area_fraction.mean('time', skipna=False),
                'min_valid_area_fraction': daily.valid_area_fraction.min('time', skipna=False),
            })
            if 'county_id' not in annual.coords or 'county_name' not in annual.coords:
                raise ValueError('Shared aggregator must preserve county_id and county_name')
            table = annual.to_dataframe().reset_index()
            table['county_id'] = table['county_id'].astype(str).str.zfill(5)
            table['school_year'] = year
            table['expected_days'] = len(expected)
            table['calendar'] = calendar_name
            table['model'], table['member'], table['scenario'] = model, member, scenario
            tables.append(table)
            if year == FIRST_YEAR or year % 10 == 0:
                print(f'{model}/{member}/{scenario}: school year {year}', flush=True)
    table = pd.concat(tables, ignore_index=True)
    columns = ['model','member','scenario','county_id','county_name','school_year',
               'school_year_mean_tasmax_c','expected_days','valid_days',
               'mean_valid_area_fraction','min_valid_area_fraction','calendar']
    table = table[columns].sort_values(['school_year','county_id']).reset_index(drop=True)
    if len(table) != 102 * (LAST_YEAR - FIRST_YEAR + 1):
        raise ValueError('Unexpected output row count')
    if table.duplicated(['school_year','county_id']).any():
        raise ValueError('Duplicate county-school-year rows')
    if not np.isfinite(table.school_year_mean_tasmax_c).any():
        raise ValueError('All school-year temperatures are missing; output not written')
    return table

def write_csv(table, target):
    temporary = None
    try:
        with tempfile.NamedTemporaryFile(dir=target.parent, suffix='.tmp', delete=False) as handle:
            temporary = Path(handle.name)
        table.to_csv(temporary, index=False, na_rep='NaN', float_format='%.8f')
        check = pd.read_csv(temporary, dtype={'county_id': str})
        if len(check) != len(table):
            raise ValueError('CSV row count changed during writing')
        np.testing.assert_allclose(check.school_year_mean_tasmax_c,
                                   table.school_year_mean_tasmax_c, atol=1e-7, equal_nan=True)
        if OVERWRITE:
            os.replace(temporary, target)
        else:
            os.link(temporary, target)  # Refuse replacement even if another process wrote it.
            temporary.unlink()
    finally:
        if temporary is not None and temporary.exists():
            temporary.unlink()


In [ ]:
jobs = discover_runs()
print(f'Discovered {len(jobs)} future tasmax runs')
if not jobs:
    raise FileNotFoundError(f'No future tasmax runs under {INPUT_ROOT}')
display(pd.DataFrame(jobs, columns=['model','member','scenario']).groupby('scenario').size().rename('runs'))
print('Output:', OUTPUT_ROOT)
print('Overwrite:', OVERWRITE)


In [ ]:
failures = []
completed = skipped = 0
selected_jobs = jobs if RUN_LIMIT is None else jobs[:RUN_LIMIT]
for model, member, scenario in tqdm(selected_jobs, desc='Park 2020 school years', unit='run'):
    target = OUTPUT_ROOT / f'Park_et_al_2020__{model}_{member}_{scenario}.csv'
    if target.exists() and not OVERWRITE:
        skipped += 1
        print('SKIPPED:', target.name)
        continue
    try:
        table = create_table(model, member, scenario)
        write_csv(table, target)
        completed += 1
        print('WROTE:', target.name,
              'missing county-years:', int(table.school_year_mean_tasmax_c.isna().sum()))
    except Exception as exc:
        failures.append({'model':model, 'member':member, 'scenario':scenario,
                         'error':f'{type(exc).__name__}: {exc}'})
        print('FAILED:', failures[-1])
print(f'Completed={completed}; skipped={skipped}; failed={len(failures)}')
if failures:
    display(pd.DataFrame(failures))


## Inspect one output
CSV is the requested delivery format. Each full run has 11,220 rows (110 school years × 102 counties). Read county FIPS as text. Existing files skipped above are not revalidated or regenerated.

In [ ]:
outputs = sorted(OUTPUT_ROOT.glob('Park_et_al_2020__*.csv'))
if outputs:
    sample = pd.read_csv(outputs[0], dtype={'county_id': str})
    print(outputs[0].name)
    display(sample.head())
    display(sample['school_year_mean_tasmax_c'].describe())
    print('Missing temperatures:', sample.school_year_mean_tasmax_c.isna().sum())
